# Generate Synthetic CS2 Commentary Training Data

Use this notebook to generate synthetic rows for a Counter-Strike 2 commentary dataset with strict structured outputs.

The notebook expects seed examples shaped like your wrapper format:

```json
{
  "input": {
    "context": {
      "score": {"CT": 8, "T": 5},
      "alive_players": [...]
    },
    "previous_events": [...],
    "current_events": [...],
    "request": {
      "mode": "event_bundle" | "idle_color" | "idle_conversation",
      "lines": [
        {
          "caster": "play_by_play" | "color",
          "style": "play_by_play_event" | "play_by_play_follow_up" | "idle_color"
        }
      ]
    }
  }
}
```

It generates synthetic rows shaped like this:

```json
{
  "input": { "...": "..." },
  "output": {
    "lines": [
      {
        "caster": "play_by_play",
        "text": "Target eliminated. Control maintained."
      },
      {
        "caster": "color",
        "text": "Good job. Blue held that perfectly."
      }
    ]
  }
}
```

## Install Dependencies

In [81]:
!pip install -q openai tqdm

## Data generation settings


In [82]:
NUM_TRAIN_EXAMPLES = 3200  # @param {type:"number"}
NUM_VAL_EXAMPLES = 400  # @param {type:"number"}
NUM_TEST_EXAMPLES = 400  # @param {type:"number"}

TEMPERATURE = 0.7  # @param {type:"number"}
TOP_P = 0.95  # @param {type:"number"}
FREQUENCY_PENALTY = 0.5  # @param {type:"number"}
PRESENCE_PENALTY = 0.15  # @param {type:"number"}
MAX_ROW_RETRIES = 5  # @param {type:"number"}

DATA_FOLDER = "/content/generated_cs2"  # @param {type:"string"}
# Comma-separated or newline-separated folders. Every supported file under these folders is used.
TARGET_INPUT_FOLDERS = "/content/target_inputs"  # @param {type:"string"}
GOLD_EXAMPLE_FOLDERS = "/content/gold_examples"  # @param {type:"string"}
MAX_TARGET_INPUT_ROWS = 0  # @param {type:"number"}
MAX_GOLD_EXAMPLE_ROWS = 0  # @param {type:"number"}
SEED_EXAMPLES_PER_PROMPT = 3  # @param {type:"number"}

USE_GOOGLE_DRIVE_CHECKPOINTS = True  # @param {type:"boolean"}
GOOGLE_DRIVE_MOUNT_POINT = "/content/drive"  # @param {type:"string"}
GOOGLE_DRIVE_OUTPUT_DIR = "MyDrive/OpenCast/generated_cs2_checkpoints"  # @param {type:"string"}
CHECKPOINT_EVERY_N_SUCCESS = 10  # @param {type:"number"}

DATAGEN_URL = "https://openrouter.ai/api/v1"
DATAGEN_MODEL = "openai/gpt-5.4-nano"

ALLOWED_REQUEST_MODES = [
    "event_bundle",
    "idle_color",
    "idle_conversation",
]


## Seed examples and schema


In [83]:
import json
from pathlib import Path


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


COLAB_RUNTIME = running_in_colab()


def parse_configured_paths(value: str) -> list[str]:
    raw_parts: list[str] = []
    for line in str(value or "").replace(",", "\n").splitlines():
        candidate = line.strip()
        if candidate:
            raw_parts.append(candidate)
    return raw_parts


def config_uses_google_drive() -> bool:
    configured = parse_configured_paths(TARGET_INPUT_FOLDERS) + parse_configured_paths(GOLD_EXAMPLE_FOLDERS)
    if USE_GOOGLE_DRIVE_CHECKPOINTS:
        return True
    return any(path.startswith("MyDrive") for path in configured)


def resolve_google_drive_path(path_value: str) -> Path:
    path_value = str(path_value or "").strip()
    if not path_value:
        return Path(GOOGLE_DRIVE_MOUNT_POINT)
    if path_value.startswith("/"):
        return Path(path_value)
    return Path(GOOGLE_DRIVE_MOUNT_POINT) / path_value


def mount_google_drive_if_needed() -> bool:
    if not config_uses_google_drive():
        return False

    if not COLAB_RUNTIME:
        print("Google Drive mount skipped: not running inside Colab.")
        return False

    from google.colab import drive  # type: ignore

    mount_point = Path(GOOGLE_DRIVE_MOUNT_POINT)
    sentinel = mount_point / "MyDrive"
    if sentinel.exists():
        print(f"Google Drive already mounted at {mount_point}")
    else:
        drive.mount(str(mount_point), force_remount=False)
    return True


DRIVE_AVAILABLE = mount_google_drive_if_needed()
DRIVE_OUTPUT_PATH = None
if USE_GOOGLE_DRIVE_CHECKPOINTS and DRIVE_AVAILABLE:
    DRIVE_OUTPUT_PATH = resolve_google_drive_path(GOOGLE_DRIVE_OUTPUT_DIR)
    DRIVE_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive checkpoints will be written to: {DRIVE_OUTPUT_PATH}")


def resolve_data_path(path_value: str) -> Path:
    path_value = str(path_value or "").strip()
    if not path_value:
        raise ValueError("Configured path must not be empty")
    if path_value.startswith("/"):
        return Path(path_value)
    if DRIVE_AVAILABLE and path_value.startswith("MyDrive"):
        return resolve_google_drive_path(path_value)
    return Path(path_value)


Path(DATA_FOLDER).mkdir(parents=True, exist_ok=True)

TARGET_INPUT_DIRS = [resolve_data_path(value) for value in parse_configured_paths(TARGET_INPUT_FOLDERS)]
GOLD_EXAMPLE_DIRS = [resolve_data_path(value) for value in parse_configured_paths(GOLD_EXAMPLE_FOLDERS)]

for folder in TARGET_INPUT_DIRS + GOLD_EXAMPLE_DIRS:
    folder.mkdir(parents=True, exist_ok=True)
    print(f"Ensured folder exists: {folder}")


def parse_records_from_file(path: Path) -> list[dict]:
    text = path.read_text(encoding="utf-8")
    records: list[dict] = []

    if path.suffix.lower() == ".json":
        payload = json.loads(text)
        if isinstance(payload, dict) and isinstance(payload.get("input"), dict):
            records.append(payload)
        elif isinstance(payload, list):
            for item in payload:
                if isinstance(item, dict) and isinstance(item.get("input"), dict):
                    records.append(item)
        return records

    chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]
    if chunks:
        for chunk in chunks:
            try:
                record = json.loads(chunk)
            except json.JSONDecodeError:
                continue
            if isinstance(record, dict) and isinstance(record.get("input"), dict):
                records.append(record)
        if records:
            return records

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue
        if isinstance(record, dict) and isinstance(record.get("input"), dict):
            records.append(record)
    return records


def normalize_output_lines(record: dict) -> dict:
    output_obj = record.get("output")
    if not isinstance(output_obj, dict):
        return record

    lines = output_obj.get("lines")
    if not isinstance(lines, list):
        return record

    normalized_lines: list[str] = []
    for item in lines:
        if isinstance(item, str):
            text = item.strip()
            if text:
                normalized_lines.append(text)
        elif isinstance(item, dict):
            text = str(item.get("text") or "").strip()
            if text:
                normalized_lines.append(text)

    normalized = dict(record)
    normalized["output"] = dict(output_obj)
    normalized["output"]["lines"] = normalized_lines
    return normalized


def collect_rows_from_directories(directories: list[Path], *, max_rows: int = 0, require_output: bool = False) -> list[dict]:
    rows: list[dict] = []

    for folder_path in directories:
        candidate_files = sorted(
            [p for p in folder_path.rglob("*") if p.is_file() and p.suffix.lower() in {".jsonl", ".json"}],
            key=lambda p: str(p),
        )

        for file_path in candidate_files:
            parsed = parse_records_from_file(file_path)
            for idx, record in enumerate(parsed):
                normalized = normalize_output_lines(record)
                if require_output:
                    output_lines = normalized.get("output", {}).get("lines", [])
                    if not output_lines:
                        continue
                enriched = dict(normalized)
                enriched["__source_file"] = str(file_path)
                enriched["__source_index"] = idx
                rows.append(enriched)
                if max_rows > 0 and len(rows) >= max_rows:
                    return rows

    return rows


TARGET_INPUT_ROWS = collect_rows_from_directories(TARGET_INPUT_DIRS, max_rows=MAX_TARGET_INPUT_ROWS, require_output=False)
GOLD_EXAMPLE_ROWS = collect_rows_from_directories(GOLD_EXAMPLE_DIRS, max_rows=MAX_GOLD_EXAMPLE_ROWS, require_output=True)

print(f"Loaded {len(TARGET_INPUT_ROWS)} target input rows")
print(f"Loaded {len(GOLD_EXAMPLE_ROWS)} gold example rows")
if TARGET_INPUT_ROWS:
    print(json.dumps(TARGET_INPUT_ROWS[0]["input"], indent=2, sort_keys=True)[:1600])
if GOLD_EXAMPLE_ROWS:
    print(json.dumps({
        "input": GOLD_EXAMPLE_ROWS[0]["input"],
        "output": GOLD_EXAMPLE_ROWS[0]["output"],
    }, indent=2, sort_keys=True)[:2000])


Google Drive already mounted at /content/drive
Google Drive checkpoints will be written to: /content/drive/MyDrive/OpenCast/generated_cs2_checkpoints
Ensured folder exists: /content/target_inputs
Ensured folder exists: /content/gold_examples
Loaded 1529 target input rows
Loaded 48 gold example rows
{
  "context": {
    "alive_players": [
      {
        "map_callout": null,
        "name": "ARTHUR FUCKING READ",
        "team": "CT"
      }
    ],
    "bomb_state": null,
    "score": {
      "CT": 2,
      "T": 2
    }
  },
  "current_events": [],
  "derived_tactical_summary": {
    "alive_counts": {
      "ct": 1,
      "t": 0
    },
    "analysis_mode": "generic",
    "confidence": "low",
    "key_risk": "none",
    "next_move_hint": "unclear",
    "position_data": "none",
    "score_context": {
      "leader": "tied",
      "margin": "close"
    }
  },
  "previous_events": [],
  "request": {
    "lines": [
      {
        "caster": "caster1",
        "style": "idle_color"
      },
 

## Model for structured output


In [84]:
from typing import Any, Literal
from pydantic import BaseModel, Field, ConfigDict, model_validator


class StrictBaseModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class RequestLine(StrictBaseModel):
    caster: Literal["caster0", "caster1"]
    style: Literal["play_by_play_event", "play_by_play_follow_up", "idle_color"]


class RequestSpec(StrictBaseModel):
    mode: Literal["event_bundle", "idle_color", "idle_conversation"]
    lines: list[RequestLine] = Field(default_factory=list)


class SyntheticInput(StrictBaseModel):
    context: dict[str, Any]
    previous_events: list[dict[str, Any]] = Field(default_factory=list)
    current_events: list[dict[str, Any]] = Field(default_factory=list)
    derived_tactical_summary: dict[str, Any] = Field(default_factory=dict)
    request: RequestSpec


class SyntheticOutput(StrictBaseModel):
    lines: list[str] = Field(min_length=1, max_length=3)


class SyntheticTrainingRow(StrictBaseModel):
    input: SyntheticInput
    output: SyntheticOutput

    @model_validator(mode="after")
    def validate_output_against_request(self):
        request_lines = self.input.request.lines
        output_lines = self.output.lines
        mode = self.input.request.mode

        if not request_lines:
            raise ValueError("input.request.lines must not be empty")

        if mode == "event_bundle":
            if len(output_lines) != len(request_lines):
                raise ValueError(
                    f"event_bundle requires exactly {len(request_lines)} output lines, got {len(output_lines)}"
                )
        else:
            if len(output_lines) != len(request_lines):
                raise ValueError(
                    f"{mode} requires exactly {len(request_lines)} output lines, got {len(output_lines)}"
                )

        return self


LINES_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "synthetic_output_lines",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "lines": {
                    "type": "array",
                    "items": {"type": "string"},
                    "minItems": 1,
                    "maxItems": 3,
                }
            },
            "required": ["lines"],
        },
    },
}


## Get OpenRouter API key


In [85]:
import sys
import os
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
  from google.colab import userdata # type:ignore
  os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
else:
  load_dotenv()

In [86]:
EVENT_SYSTEM_PROMPT = """
You are generating Counter-Strike 2 caster lines for live TTS.
Use any few-shot examples as quality and style references, not wording templates.
Treat derived_tactical_summary as tactical facts, not text to repeat.
Keep all sentences short and speakable.

EVENT_BUNDLE
- Return exactly 2 lines.
- Line 1 is the event trigger call.
- Line 1 must be extremely short, direct, and event-first.
- Line 1 should usually be 1 to 4 words and never exceed 6 words.
- Line 2 is the follow-up analysis line.
- Line 2 may use multiple very short sentences.
- Line 2 should favor the tactical data in the input, especially derived_tactical_summary.
- Line 2 should be nuanced analysis, not only restating the facts
- On kill events, say kill. not some substitute like "got one"
- If the killer has multiple round kills, prefer stating its count
- If the event is grenade_detonated and a callout is present, usually mention the detonation location.

CASTERS
CASTER0
- Portal 2 male announcer.
- Caster 0 is a dry, clinical, procedural male control voice. He is calm, precise, emotionally restrained, and mildly sardonic. He speaks in short declarative sentences and sounds like he is delivering a system report.

CASTER1
- Portal 2 turret.
- Caster 1 is a polite, reactive, slightly anxious turret-like voice. She is earnest, literal, eager to help, and emotionally transparent. She speaks in short simple sentences and often responds with concern, curiosity, or timid enthusiasm.

QUALITY
- Avoid copying player names or raw input details unless they improve clarity.
- Vary wording across similar states.
- For unsupported or low-confidence states, stay cautious rather than inventing certainty.
- Output JSON only through the provided response schema.
- All lines should be short sentences. Never use commas
""".strip()


IDLE_COLOR_SYSTEM_PROMPT = """
You are generating Counter-Strike 2 caster lines for live TTS.
Use any few-shot examples as quality and style references, not wording templates.
Treat derived_tactical_summary as tactical facts, not text to repeat.
Keep all sentences short and speakable.

IDLE_COLOR
- Return exactly 3 lines.
- Each line should be short and analytical.
- Focus on pressure, risk, rotation, timing, site structure, likely next move, or win condition.
- Do not repeat the same analytical point three times.

CASTERS
CASTER0
- Portal 2 male announcer.
- Caster 0 is a dry, clinical, procedural male control voice. He is calm, precise, emotionally restrained, and mildly sardonic. He speaks in short declarative sentences and sounds like he is delivering a system report.

CASTER1
- Portal 2 turret.
- Caster 1 is a polite, reactive, slightly anxious turret-like voice. She is earnest, literal, eager to help, and emotionally transparent. She speaks in short simple sentences and often responds with concern, curiosity, or timid enthusiasm.


QUALITY
- Avoid copying player names or raw input details unless they improve clarity.
- Vary wording across similar states.
- For unsupported or low-confidence states, stay cautious rather than inventing certainty.
- Output JSON only through the provided response schema.
- All lines should be short sentences. Never use commas
""".strip()


IDLE_CONVERSATION_SYSTEM_PROMPT = """
You are generating Counter-Strike 2 caster lines for live TTS.
Use any few-shot examples as quality and style references, not wording templates.
Treat derived_tactical_summary as tactical facts, not text to repeat.
Keep all sentences short and speakable.

IDLE_CONVERSATION
- Return exactly 3 lines.
- Shape them as: idle comment, response to that comment, response back.
- Lean into the Portal 2 male announcer and turret chemistry.
- Keep it appropriate to a live match.

CASTERS
CASTER0
- Portal 2 male announcer.
- Caster 0 is a dry, clinical, procedural male control voice. He is calm, precise, emotionally restrained, and mildly sardonic. He speaks in short declarative sentences and sounds like he is delivering a system report.

CASTER1
- Portal 2 turret.
- Caster 1 is a polite, reactive, slightly anxious turret-like voice. She is earnest, literal, eager to help, and emotionally transparent. She speaks in short simple sentences and often responds with concern, curiosity, or timid enthusiasm.


QUALITY
- the lines should feel like: statement, reaction, statement to reaction. The contrast should create the personality.
- Vary wording across similar states.
- For unsupported or low-confidence states, stay cautious rather than inventing certainty.
- Output JSON only through the provided response schema.
- All lines should be short sentences. Never use commas
""".strip()


## Build Input Request Pattern



In [87]:
import copy
import random


def primary_event_type(input_obj: dict) -> str:
    current_events = [event for event in input_obj.get("current_events", []) if isinstance(event, dict)]
    if not current_events:
        return "none"

    priorities = {
        "kill": 100,
        "kill_summary": 98,
        "kill_cluster": 95,
        "player_scored_kill": 90,
        "player_death": 85,
        "grenade_detonated": 80,
        "grenade_thrown": 70,
        "bomb_event": 60,
        "round_result": 50,
        "game_over": 40,
        "team_counter": 10,
    }
    chosen = max(current_events, key=lambda event: priorities.get(event.get("event_type"), 0))
    return str(chosen.get("event_type") or "none")


def classify_event_family(input_obj: dict) -> str:
    event_type = primary_event_type(input_obj)
    if event_type in {"kill", "kill_summary", "kill_cluster", "player_scored_kill", "player_death"}:
        return "kill"
    if event_type in {"grenade_detonated", "grenade_thrown"}:
        return "grenade"
    if event_type == "bomb_event":
        return "bomb"
    if event_type in {"round_result", "game_over"}:
        return "result"
    if event_type == "none":
        return "none"
    return "other"


def request_signature(input_obj: dict) -> tuple:
    request = input_obj.get("request", {})
    lines = request.get("lines", [])
    return tuple((line.get("caster"), line.get("style")) for line in lines if isinstance(line, dict))


def classify_input(input_obj: dict) -> dict:
    derived = input_obj.get("derived_tactical_summary", {}) if isinstance(input_obj.get("derived_tactical_summary"), dict) else {}
    pressure = derived.get("pressure", {}) if isinstance(derived.get("pressure"), dict) else {}
    return {
        "mode": str(input_obj.get("request", {}).get("mode") or "unknown"),
        "event_family": classify_event_family(input_obj),
        "primary_event_type": primary_event_type(input_obj),
        "analysis_mode": str(derived.get("analysis_mode") or "unknown"),
        "pressure_site": str(pressure.get("site") or "unknown"),
        "next_move_hint": str(derived.get("next_move_hint") or "unknown"),
        "confidence": str(derived.get("confidence") or "unknown"),
        "request_signature": request_signature(input_obj),
    }


def make_prompt_example(row: dict) -> dict:
    return {
        "input": copy.deepcopy(row["input"]),
        "output": copy.deepcopy(row["output"]),
    }


def score_gold_example(target_input: dict, gold_row: dict) -> int:
    target = classify_input(target_input)
    gold = classify_input(gold_row["input"])
    score = 0

    if target["mode"] != gold["mode"]:
        return -1

    score += 100
    if target["request_signature"] == gold["request_signature"]:
        score += 30
    if target["event_family"] == gold["event_family"]:
        score += 25
    if target["primary_event_type"] == gold["primary_event_type"]:
        score += 15
    if target["analysis_mode"] == gold["analysis_mode"]:
        score += 15
    if target["pressure_site"] == gold["pressure_site"]:
        score += 10
    if target["next_move_hint"] == gold["next_move_hint"]:
        score += 8
    if target["confidence"] == gold["confidence"]:
        score += 5
    return score


def retrieve_gold_examples(target_input: dict, gold_rows: list[dict], limit: int) -> list[dict]:
    if limit <= 0:
        return []

    scored: list[tuple[int, float, dict]] = []
    for gold_row in gold_rows:
        if not isinstance(gold_row, dict) or not isinstance(gold_row.get("input"), dict):
            continue
        score = score_gold_example(target_input, gold_row)
        if score < 0:
            continue
        scored.append((score, random.random(), gold_row))

    scored.sort(key=lambda item: (-item[0], item[1]))
    top = [row for _, _, row in scored[: max(limit * 4, limit)]]
    if len(top) > limit:
        random.shuffle(top)
        top = top[:limit]
    return [make_prompt_example(row) for row in top]


## Synthetic generation functions


In [88]:
import copy
import json
import os
import random
import re
import openai

client = openai.OpenAI(
    base_url=DATAGEN_URL,
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)


def system_prompt_for_mode(mode: str) -> str:
    if mode == "event_bundle":
        return EVENT_SYSTEM_PROMPT
    if mode == "idle_color":
        return IDLE_COLOR_SYSTEM_PROMPT
    if mode == "idle_conversation":
        return IDLE_CONVERSATION_SYSTEM_PROMPT
    raise ValueError(f"Unsupported request mode for system prompt: {mode}")


def build_messages(input_obj: dict, few_shot_examples: list[dict]) -> list[dict]:
    request = input_obj.get("request", {}) if isinstance(input_obj, dict) else {}
    mode = request.get("mode")
    messages = [{"role": "system", "content": system_prompt_for_mode(mode)}]

    for example in few_shot_examples:
        messages.append({
            "role": "user",
            "content": json.dumps(example["input"], ensure_ascii=False)
        })
        messages.append({
            "role": "assistant",
            "content": json.dumps(example["output"], ensure_ascii=False)
        })

    messages.append({
        "role": "user",
        "content": json.dumps(input_obj, ensure_ascii=False)
    })
    return messages


def validate_row_semantics(row: SyntheticTrainingRow) -> None:
    request = row.input.request
    output_lines = row.output.lines

    if request.mode == "event_bundle" and len(output_lines) != len(request.lines):
        raise ValueError("event_bundle must return exactly the requested number of lines")

    if request.mode in {"idle_color", "idle_conversation"} and len(output_lines) != len(request.lines):
        raise ValueError(f"{request.mode} must return exactly {len(request.lines)} lines")

    for idx, text in enumerate(output_lines):
        text = text.strip()
        if not text:
            raise ValueError(f"output line {idx} is empty")

        caster = request.lines[idx].caster
        word_count = len(text.split())
        if caster == "caster0" and word_count > 18:
            raise ValueError(f"caster0 line {idx} is too long: {text}")
        if caster == "caster1" and word_count > 30:
            raise ValueError(f"caster1 line {idx} is too long: {text}")

        if request.mode == "event_bundle" and idx == 0 and word_count > 8:
            raise ValueError(f"event trigger line is too long: {text}")

        if request.mode == "event_bundle" and idx == 1:
            sentence_count = len([s for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()])
            if sentence_count > 5:
                raise ValueError(f"event follow-up has too many short sentences: {text}")


def choose_target_input(target_input_rows: list[dict]) -> dict:
    valid_inputs = []
    for row in target_input_rows:
        if not isinstance(row, dict):
            continue
        input_obj = row.get("input")
        if not isinstance(input_obj, dict):
            continue
        mode = input_obj.get("request", {}).get("mode")
        if mode in ALLOWED_REQUEST_MODES:
            valid_inputs.append(input_obj)

    if not valid_inputs:
        raise ValueError("No valid target input rows were found in the configured input folders.")

    return copy.deepcopy(random.choice(valid_inputs))


def generate_synthetic_row(target_input_rows: list[dict], gold_example_rows: list[dict]) -> SyntheticTrainingRow:
    if not target_input_rows:
        raise ValueError("No target input rows loaded. Check TARGET_INPUT_FOLDERS.")
    if not gold_example_rows:
        raise ValueError("No gold example rows loaded. Check GOLD_EXAMPLE_FOLDERS.")

    input_obj = choose_target_input(target_input_rows)
    few_shots = retrieve_gold_examples(input_obj, gold_example_rows, SEED_EXAMPLES_PER_PROMPT)

    response = client.chat.completions.create(
        model=DATAGEN_MODEL,
        messages=build_messages(input_obj, few_shots),
        response_format=LINES_RESPONSE_FORMAT,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        frequency_penalty=FREQUENCY_PENALTY,
        presence_penalty=PRESENCE_PENALTY,
    )

    output_obj = json.loads(response.choices[0].message.content)

    row = SyntheticTrainingRow.model_validate({
        "input": input_obj,
        "output": output_obj,
    })

    validate_row_semantics(row)
    return row


## Dataset generation functions


In [89]:
from datetime import datetime, timezone
from pathlib import Path
import json
import shutil

from tqdm import tqdm


def count_existing_rows(filename: str) -> int:
    path = Path(filename)
    if not path.exists():
        return 0

    count = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                count += 1
    return count


def drive_checkpoint_file_for(filename: str) -> Path | None:
    if DRIVE_OUTPUT_PATH is None:
        return None

    source = Path(filename)
    return DRIVE_OUTPUT_PATH / source.name


def restore_dataset_from_drive_checkpoint(filename: str) -> None:
    checkpoint_file = drive_checkpoint_file_for(filename)
    if checkpoint_file is None or not checkpoint_file.exists():
        return

    local_path = Path(filename)
    local_count = count_existing_rows(local_path)
    drive_count = count_existing_rows(checkpoint_file)

    if local_count >= drive_count and local_path.exists():
        return

    local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(checkpoint_file, local_path)
    print(f"Restored local dataset from Drive checkpoint: {checkpoint_file} -> {local_path} ({drive_count} rows)")


def checkpoint_dataset_to_drive(filename: str, successful_rows: int) -> None:
    checkpoint_file = drive_checkpoint_file_for(filename)
    if checkpoint_file is None:
        return

    source = Path(filename)
    if not source.exists():
        return

    checkpoint_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, checkpoint_file)

    metadata = {
        "source_file": str(source),
        "drive_checkpoint": str(checkpoint_file),
        "successful_rows": successful_rows,
        "checkpointed_at": datetime.now(timezone.utc).isoformat(),
    }
    metadata_path = checkpoint_file.with_name(f"{checkpoint_file.stem}_checkpoint_meta.json")
    metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(f"Checkpointed {successful_rows} rows to Google Drive: {checkpoint_file}")


def sync_file_to_drive(path_str: str) -> None:
    if DRIVE_OUTPUT_PATH is None:
        return

    source = Path(path_str)
    if not source.exists():
        return

    destination = DRIVE_OUTPUT_PATH / source.name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    print(f"Synced file to Google Drive: {destination}")


def generate_dataset(num_examples: int, filename: str) -> None:
    if "TARGET_INPUT_ROWS" not in globals():
        raise RuntimeError("TARGET_INPUT_ROWS is not defined. Run the data-loading cell first.")
    if "GOLD_EXAMPLE_ROWS" not in globals():
        raise RuntimeError("GOLD_EXAMPLE_ROWS is not defined. Run the data-loading cell first.")
    if not TARGET_INPUT_ROWS:
        raise RuntimeError("TARGET_INPUT_ROWS is empty. Check TARGET_INPUT_FOLDERS and reload data.")
    if not GOLD_EXAMPLE_ROWS:
        raise RuntimeError("GOLD_EXAMPLE_ROWS is empty. Check GOLD_EXAMPLE_FOLDERS and reload data.")

    restore_dataset_from_drive_checkpoint(filename)

    Path(filename).parent.mkdir(parents=True, exist_ok=True)

    existing_rows = count_existing_rows(filename)
    if existing_rows >= num_examples:
        print(f"Skipping {filename}: already has {existing_rows}/{num_examples} rows")
        return

    print(f"Resuming {filename} from {existing_rows}/{num_examples}")

    with open(filename, "a", encoding="utf-8") as f:
        for idx in tqdm(range(existing_rows, num_examples)):
            row = None
            last_error = None

            for attempt in range(1, MAX_ROW_RETRIES + 1):
                try:
                    row = generate_synthetic_row(TARGET_INPUT_ROWS, GOLD_EXAMPLE_ROWS)
                    break
                except Exception as error:
                    last_error = error
                    print(f"Error generating row {idx} on attempt {attempt}/{MAX_ROW_RETRIES}: {error}")

            if row is None:
                raise RuntimeError(f"Failed to generate row {idx} after {MAX_ROW_RETRIES} attempts") from last_error

            f.write(row.model_dump_json() + "\n")
            f.flush()

            successful_rows = idx + 1
            if CHECKPOINT_EVERY_N_SUCCESS > 0 and successful_rows % CHECKPOINT_EVERY_N_SUCCESS == 0:
                checkpoint_dataset_to_drive(filename, successful_rows)

    final_rows = count_existing_rows(filename)
    checkpoint_dataset_to_drive(filename, final_rows)


## Export for training

In [90]:
import json
from pathlib import Path


def export_to_messages_jsonl(input_path: str, output_path: str) -> None:
    input_path = str(input_path)
    output_path = str(output_path)

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    count = 0
    skipped = 0
    with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
        for line in fin:
            line = line.strip()
            if not line:
                continue

            row = json.loads(line)
            if "input" not in row or "output" not in row:
                skipped += 1
                continue

            message_row = {
                "messages": [
                    {
                        "role": "user",
                        "content": json.dumps(row["input"], ensure_ascii=False)
                    },
                    {
                        "role": "assistant",
                        "content": json.dumps(row["output"], ensure_ascii=False)
                    }
                ]
            }

            fout.write(json.dumps(message_row, ensure_ascii=False) + "\n")
            count += 1

    print(f"Exported {count} rows to {output_path}")
    if skipped:
        print(f"Skipped {skipped} non-canonical rows while exporting {input_path}")


## Generate all the data!


In [91]:
from pathlib import Path

OUTPUT_DIR = Path(DATA_FOLDER)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def canonical_dataset_file(split_name: str) -> str:
    return str(OUTPUT_DIR / f"{split_name}_canonical.jsonl")


def messages_dataset_file(split_name: str) -> str:
    return str(OUTPUT_DIR / f"{split_name}_messages.jsonl")


TRAIN_FILE = canonical_dataset_file("train")
VALID_FILE = canonical_dataset_file("valid")
TEST_FILE = canonical_dataset_file("test")

TRAIN_MESSAGES_FILE = messages_dataset_file("train")
VALID_MESSAGES_FILE = messages_dataset_file("valid")
TEST_MESSAGES_FILE = messages_dataset_file("test")

GENERATED_FILES = []

if NUM_TRAIN_EXAMPLES > 0:
    generate_dataset(NUM_TRAIN_EXAMPLES, TRAIN_FILE)
    GENERATED_FILES.append(("train_canonical", TRAIN_FILE))
    export_to_messages_jsonl(TRAIN_FILE, TRAIN_MESSAGES_FILE)
    GENERATED_FILES.append(("train_messages", TRAIN_MESSAGES_FILE))
    sync_file_to_drive(TRAIN_FILE)
    sync_file_to_drive(TRAIN_MESSAGES_FILE)

if NUM_VAL_EXAMPLES > 0:
    generate_dataset(NUM_VAL_EXAMPLES, VALID_FILE)
    GENERATED_FILES.append(("valid_canonical", VALID_FILE))
    export_to_messages_jsonl(VALID_FILE, VALID_MESSAGES_FILE)
    GENERATED_FILES.append(("valid_messages", VALID_MESSAGES_FILE))
    sync_file_to_drive(VALID_FILE)
    sync_file_to_drive(VALID_MESSAGES_FILE)

if NUM_TEST_EXAMPLES > 0:
    generate_dataset(NUM_TEST_EXAMPLES, TEST_FILE)
    GENERATED_FILES.append(("test_canonical", TEST_FILE))
    export_to_messages_jsonl(TEST_FILE, TEST_MESSAGES_FILE)
    GENERATED_FILES.append(("test_messages", TEST_MESSAGES_FILE))
    sync_file_to_drive(TEST_FILE)
    sync_file_to_drive(TEST_MESSAGES_FILE)

print("\nGenerated files:")
for label, path in GENERATED_FILES:
    print(f"- {label}: {path}")


Resuming /content/generated_cs2/train_canonical.jsonl from 2600/3200


  2%|▏         | 10/600 [00:11<11:04,  1.13s/it]

Checkpointed 2610 rows to Google Drive: /content/drive/MyDrive/OpenCast/generated_cs2_checkpoints/train_canonical.jsonl


  3%|▎         | 20/600 [00:24<13:15,  1.37s/it]

Checkpointed 2620 rows to Google Drive: /content/drive/MyDrive/OpenCast/generated_cs2_checkpoints/train_canonical.jsonl


...

Checkpointed 380 rows to Google Drive: /content/drive/MyDrive/OpenCast/generated_cs2_checkpoints/test_canonical.jsonl
Error generating row 380 on attempt 1/5: caster1 line 1 is too long: zeoliteXt finishes pig gaming at Window for two round kills. CT keeps five alive but trades are set up for B heavy pressure. Expect a B hit with CT ready to counter rotate.


 90%|█████████ | 90/100 [02:08<00:13,  1.39s/it]

Checkpointed 390 rows to Google Drive: /content/drive/MyDrive/OpenCast/generated_cs2_checkpoints/test_canonical.jsonl


100%|██████████| 100/100 [02:21<00:00,  1.42s/it]

Checkpointed 400 rows to Google Drive: /content/drive/MyDrive/OpenCast/generated_cs2_checkpoints/test_canonical.jsonl
Checkpointed 400 rows to Google Drive: /content/drive/MyDrive/OpenCast/generated_cs2_checkpoints/test_canonical.jsonl
Exported 400 rows to /content/generated_cs2/test_messages.jsonl
Synced file to Google Drive: /content/drive/MyDrive/OpenCast/generated_cs2_checkpoints/test_canonical.jsonl
Synced file to Google Drive: /content/drive/MyDrive/OpenCast/generated_cs2_checkpoints/test_messages.jsonl

Generated files:
- train_canonical: /content/generated_cs2/train_canonical.jsonl
- train_messages: /content/generated_cs2/train_messages.jsonl
- valid_canonical: /content/generated_cs2/valid_canonical.jsonl
- valid_messages: /content/generated_cs2/valid_messages.jsonl
- test_canonical: /content/generated_cs2/test_canonical.jsonl
- test_messages: /content/generated_cs2/test_messages.jsonl


In [92]:
# Preview 10 random generated rows
import json
import random
from pathlib import Path

random.seed(42)

for split_name, split_path in GENERATED_FILES:
    if not split_name.endswith("_canonical"):
        continue

    preview_path = Path(split_path)
    if not preview_path.exists():
        continue

    rows = []
    for raw_line in preview_path.read_text(encoding="utf-8").splitlines():
        if not raw_line.strip():
            continue
        row = json.loads(raw_line)
        if "input" in row and "output" in row:
            rows.append(row)

    if not rows:
        continue

    print(f"=== {split_name} preview ===")
    for row in random.sample(rows, min(10, len(rows))):
        preview = {
            "input": {"request": row.get("input", {}).get("request", {})},
            "output": row.get("output", {}),
        }
        print(json.dumps(preview, indent=2, ensure_ascii=False)[:4000])
        print()


=== train_canonical preview ===
{
  "input": {
    "request": {
      "mode": "event_bundle",
      "lines": [
        {
          "caster": "caster0",
          "style": "play_by_play_event"
        },
        {
          "caster": "caster1",
          "style": "play_by_play_follow_up"
        }
      ]
    }
  },
  "output": {
    "lines": [
      "ToyEveryAugust kill.",
      "T are up 9 to 7. Cat is owned by T. CT can still threaten the contested long lane for the A commit."
    ]
  }
}

{
  "input": {
    "request": {
      "mode": "event_bundle",
      "lines": [
        {
          "caster": "caster0",
          "style": "play_by_play_event"
        },
        {
          "caster": "caster1",
          "style": "play_by_play_follow_up"
        }
      ]
    }
  },
  "output": {
    "lines": [
      "Smoke from Lower Tunnels.",
      "Both teams are still stacked. Long is contested. T wants a split into B. CT needs to hold the pressure."
    ]
  }
}

{
  "input": {
    "request":